# graphArt_LM: does training on both T4s work? (smoke test, no real training)

Three checks, about 30 minutes, capped at 1 hour. Nothing is kept but the log:
1. Kaggle gave the session two GPUs.
2. The same few training steps on one GPU and then on two: the seconds per step of each,
   the time a full epoch would take on two, and that the loss still falls on two.
3. The time guard on two GPUs: a short run whose 3-minute budget must stop both at the
   same step, save the adapter, and exit (a hang is caught and reported).

The verdict is printed at the end, after `##### VERDICT`.

**Before running**: *GPU T4 x2* (both are used), *Internet on*. No inputs. Then
*Save Version -> Save & Run All (Commit)*.

Generated by `scripts/make_kaggle_notebook.py --two-gpu-smoke`. Edit that script, not this file.

In [ ]:
# BASE_MODEL   the model the real run will tune.
# TRAIN_ARGS   the full-prompt two-step data, as the real run uses it (--max-len 6144, where
#              the last run's memory check landed; the few longer examples are left out).
# SMOKE_STEPS  steps timed on one GPU, then on two -- the same effective batch both times.
# GUARD_BUDGET_HOURS  the time budget of the guard test (3 min): it must stop both GPUs at
#              the same step, save, and exit, not hang.
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"; TRAIN_ARGS = ["--two-step", "--full-prompts", "--train-split", "train/8,large_train", "--val-split", "val/8,large_val/2", "--max-len", "6144", "--drop-over-max"]; SMOKE_STEPS = 8; EFFECTIVE_BATCH = 16; GUARD_BUDGET_HOURS = 0.05; TIME_BUDGET_HOURS = 1.0; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "a8ed39ebbe5046821b9aeea0c09c0f1696ab75907f93295304c1398f0b756268"
BUNDLE = (
    "..."
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install the pinned libraries

In [ ]:
from importlib.metadata import PackageNotFoundError, version

PINS = {'transformers': '5.20.2', 'peft': '0.21.2', 'accelerate': '1.15.0'}

def pinned(package, wanted):
    try:
        return version(package) == wanted
    except PackageNotFoundError:
        return False

missing = [f"{p}=={v}" for p, v in PINS.items() if not pinned(p, v)]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# peft probes two optional quantisation libraries when it builds LoRA layers, and
# raises -- rather than skipping them -- if they are installed but older than it
# supports. Kaggle ships torchao 0.10 (peft wants >= 0.16), which killed a run. We
# use neither, so remove them if present.
for pkg in ("torchao", "gptqmodel"):
    try:
        print(f"removing {pkg} {version(pkg)} (peft rejects old versions; not used here)", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", pkg], check=True)
    except PackageNotFoundError:
        pass

subprocess.run([sys.executable, "-c", "import torch, transformers, peft; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
                "'| transformers', transformers.__version__, '| peft', peft.__version__)"])

## 3. Download the base model

In [ ]:
# Its own step with a time limit: a stalled download inside training would otherwise
# sit until the session cap. ~3 GB; a few minutes on Kaggle's network.
FETCH = ("from huggingface_hub import snapshot_download; "
         f"print('downloaded to', snapshot_download({BASE_MODEL!r}))")
for attempt in (1, 2):
    try:
        subprocess.run([sys.executable, "-c", FETCH], check=True, timeout=1800)
        break
    except (subprocess.TimeoutExpired, subprocess.CalledProcessError) as exc:
        print(f"download attempt {attempt} failed: {exc}", flush=True)
else:
    raise SystemExit("could not download the base model -- stopping before any training")

## 4. Two GPUs?

In [ ]:
import re

CHECKS = {}
n = int(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
                       capture_output=True, text=True).stdout.strip() or 0)
print(f"{n} GPU(s) visible", flush=True)
CHECKS["two GPUs"] = n >= 2
if n < 2:
    raise SystemExit("this session has fewer than two GPUs -- set the accelerator to GPU T4 x2")

def run(cmd, env, timeout):
    # Run, echo the output live, and return (exit code, output, seconds); exit None on a hang.
    t, lines = time.time(), []
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            lines.append(line)
            if time.time() - t > timeout:
                raise subprocess.TimeoutExpired(cmd, timeout)
        code = proc.wait(timeout=max(timeout - (time.time() - t), 1))
    except subprocess.TimeoutExpired:
        proc.kill()
        code = None
    return code, "".join(lines), time.time() - t

def s_per_step(out):
    m = re.search(r"smoke: loss ([\d.]+) -> ([\d.]+) over (\d+) steps, ([\d.]+) s/step -> (\w+)", out)
    return (float(m.group(4)), float(m.group(1)), float(m.group(2)), m.group(5)) if m else None

ENV = dict(os.environ, HF_HUB_OFFLINE="1", PYTHONUNBUFFERED="1")
TRAINER = ["scripts/train_lora.py", "--base", BASE_MODEL, *TRAIN_ARGS]
TORCHRUN = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node", "2"]

## 5. The same steps on one GPU, then on two

In [ ]:
code1, out1, _ = run([sys.executable, *TRAINER, "--out", "lora/one", "--smoke-steps", str(SMOKE_STEPS),
                      "--grad-accum", str(EFFECTIVE_BATCH)], dict(ENV, CUDA_VISIBLE_DEVICES="0"), 1800)
one = s_per_step(out1)
code2, out2, _ = run([*TORCHRUN, *TRAINER, "--out", "lora/two", "--smoke-steps", str(SMOKE_STEPS),
                      "--grad-accum", str(EFFECTIVE_BATCH // 2)], dict(ENV, CUDA_VISIBLE_DEVICES="0,1"), 1800)
two = s_per_step(out2)
CHECKS["one GPU learns"] = code1 == 0 and bool(one) and one[3] == "OK"
CHECKS["two GPUs run and learn"] = code2 == 0 and bool(two) and two[3] == "OK"
examples = re.search(r"(\d+) train / \d+ val examples", out1 or out2)
EXAMPLES = int(examples.group(1)) if examples else None
print(f"one GPU: {one}  |  two GPUs: {two}  (s/step, first loss, last loss, verdict)", flush=True)

## 6. The time guard on two GPUs

In [ ]:
# 160 programs is 10 steps; a 3-minute budget stops it a few steps in. Both GPUs must stop
# at the same step, evaluate, save and exit. 20 minutes means it hung.
code3, out3, secs3 = run([*TORCHRUN, *TRAINER, "--out", "lora/guard", "--epochs", "1", "--limit-train", "160",
                          "--limit-val", "8", "--grad-accum", str(EFFECTIVE_BATCH // 2), "--eval-steps", "40",
                          "--time-budget-hours", str(GUARD_BUDGET_HOURS)],
                         dict(ENV, CUDA_VISIBLE_DEVICES="0,1"), 1200)
summary = REPO / "lora" / "guard" / "train_summary.json"
saved = json.loads(summary.read_text()) if summary.exists() else {}
stops = re.findall(r"time budget spent at step (\d+)", out3)
CHECKS["guard stops both GPUs together and saves"] = (code3 == 0 and saved.get("gpus") == 2
                                                      and len(set(stops)) == 1 and len(stops) == 2)
print(f"exit {code3} after {secs3:.0f} s; stop messages at steps {stops}; summary: "
      f"{ {k: saved.get(k) for k in ('gpus', 'steps', 'effective_batch', 'best_eval_loss')} }", flush=True)

## 7. Verdict

In [ ]:
print("\n##### VERDICT")
for name, ok in CHECKS.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
if one and two:
    steps = -(-EXAMPLES // EFFECTIVE_BATCH) if EXAMPLES else None
    print(f"  one GPU {one[0]:.1f} s/step, two GPUs {two[0]:.1f} s/step: {one[0] / two[0]:.2f}x faster")
    if steps:
        print(f"  a full epoch ({EXAMPLES} examples, {steps} steps of {EFFECTIVE_BATCH}): ~{steps * one[0] / 60:.0f} min "
              f"on one GPU, ~{steps * two[0] / 60:.0f} min on two")
print(f"  done in {(time.time() - START) / 60:.0f} min")
(WORK / "two_gpu_smoke.json").write_text(json.dumps({"checks": CHECKS, "one_gpu": one, "two_gpus": two,
                                                      "examples": EXAMPLES}, indent=1))